In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pylahman
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load the Teams table directly using pylahman
teams = pylahman.Teams()

# Preview structure
print(f"Loaded {len(teams)} rows and {len(teams.columns)} columns.")
teams.head()

In [ ]:
# Filter to the Modern Era (1901–present)
modern_teams = teams[teams["yearID"] >= 1901].copy()

# Feature engineering for exploratory analysis
modern_teams["WinPct"] = (modern_teams["W"] / modern_teams["G"]).round(3)
modern_teams["RunDiff"] = modern_teams["R"] - modern_teams["RA"]

modern_teams[
    ["yearID", "teamID", "name", "G", "W", "L", "WinPct", "RunDiff"]
].head()

In [ ]:
# 1. Run Differential vs. Wins
plt.figure(figsize=(8, 5))
sns.regplot(
    data=modern_teams,
    x="RunDiff",
    y="W",
    scatter_kws={"alpha": 0.25, "color": "steelblue"},
    line_kws={"color": "darkred"},
)
plt.axhline(81, color="gray", linestyle="--", linewidth=0.8)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.title("Wins vs. Run Differential (1901–Present)")
plt.xlabel("Run Differential (R - RA)")
plt.ylabel("Total Wins")
plt.tight_layout()
plt.show()

# 2. League-wide Home Run Evolution
hr_trend = modern_teams.groupby("yearID")["HR"].mean().reset_index()
plt.figure(figsize=(8, 5))
sns.lineplot(data=hr_trend, x="yearID", y="HR", color="forestgreen")
plt.title("Average Team Home Runs per Season (1901–Present)")
plt.xlabel("Season")
plt.ylabel("Average HR per Team")
plt.tight_layout()
plt.show()

# 3. Top 10 Single-Season Win Totals
top_10 = modern_teams.sort_values(by="W", ascending=False)[
    ["yearID", "name", "W", "L", "WinPct", "RunDiff", "WSWin"]
].head(10)

top_10

In [ ]:
# 1. Clean missing values in postseason indicator flags
div_win = modern_teams["DivWin"].fillna("N") == "Y"
wc_win = modern_teams["WCWin"].fillna("N") == "Y"

# Filter to modern division/playoff era (1969+), exclude 1994
playoffs = modern_teams[
    (modern_teams["yearID"] >= 1969)
    & (modern_teams["yearID"] != 1994)
    & (div_win | wc_win)
].copy()

# 2. Target and feature engineering
playoffs["WS_Winner"] = (
    playoffs["WSWin"].fillna("N").apply(lambda x: 1 if x == "Y" else 0)
)
playoffs["R_per_G"] = playoffs["R"] / playoffs["G"]
playoffs["RA_per_G"] = playoffs["RA"] / playoffs["G"]
playoffs["HR_per_G"] = playoffs["HR"] / playoffs["G"]

features = ["WinPct", "RunDiff", "ERA", "HR_per_G", "R_per_G", "RA_per_G"]
target = "WS_Winner"

playoffs = playoffs.dropna(subset=features + [target])

# 3. Temporal Train/Test split: Train on 1969–2010, Test on 2011–present
train = playoffs[playoffs["yearID"] <= 2010].copy()
test = playoffs[playoffs["yearID"] > 2010].copy()

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# 4. Fit Logistic Regression pipeline
model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(class_weight="balanced", random_state=42)),
    ]
)
model.fit(X_train, y_train)

# 5. Predict probabilities
test["pred_prob"] = model.predict_proba(X_test)[:, 1]

# Postseason discrimination metric
test_auc = roc_auc_score(y_test, test["pred_prob"])
print(f"Postseason Test ROC-AUC: {test_auc:.3f}")

In [ ]:
# Identify the team with the highest modeled probability for each season
predicted_champs = (
    test.sort_values(["yearID", "pred_prob"], ascending=[True, False])
    .groupby("yearID")
    .first()
    .reset_index()
)

results_table = predicted_champs[
    ["yearID", "name", "pred_prob", "WS_Winner"]
].rename(
    columns={
        "yearID": "Year",
        "name": "Predicted Favorite",
        "pred_prob": "Win Probability",
        "WS_Winner": "Actually Won WS",
    }
)

correct_picks = results_table["Actually Won WS"].sum()
total_seasons = len(results_table)

print(
    f"Favorite Pick Accuracy: {correct_picks}/{total_seasons} ({correct_picks / total_seasons:.1%})\n"
)
results_table